In [1]:
import numpy as np
from QuditGates import QuditGates
from QuditCircuit import QuditCircuit
from helper_functions import *

## Initializations

In [2]:
dimension = 3
number_S_qudits = 3 # Number of S qudits in the circuit
number_qudits = 1 + 2*number_S_qudits # Total number of qudits in the circuit 

gates = QuditGates(dimension) # Initialize the QuditGtes class

We will initialize the qudit circuit where we will apply the operators as described by equations (20) and (22) in the paper.

In [3]:
qc = QuditCircuit(number_qudits,dimension)

print("Initial state of the circuit:")
qc.display_state()

Initial state of the circuit:
|0000000> : (1+0j)


We will initialize the qudit circuit where we will apply the operators as described in Section IV of the paper.


In [4]:
qc_gates = QuditCircuit(number_qudits,dimension)

print("Initial state of the gate circuit:")
qc_gates.display_state()

Initial state of the gate circuit:
|0000000> : (1+0j)


## Initialize the circuit

We follow the following structure of the qudits in the circuit

$
    q_0 \mapsto |\psi\rangle_A\\
    q_1 \mapsto |\psi\rangle_{S_1}\\
    q_2 \mapsto |\psi\rangle_{N_1}\\
    \dots\\
    q_{2n} \mapsto |\psi\rangle_{S_{n}}\\
    q_{2n+1} \mapsto |\psi\rangle_{N_{n}}\\
$


We initialize the data qudit $|\psi\rangle_{A}$ in each of the two circuits.

In [5]:
qc.apply_gate(gates.get_I_gate(), [0]) # Apply identity gate to the first qudit
qc_gates.apply_gate(gates.get_I_gate(), [0]) # Apply identity gate to the first qudit


We initialize all the pairs of $(|\psi\rangle_{S_i},|\psi\rangle_{N_i})$ to the generalized Bell state $\frac{1}{\sqrt{d}}\sum_{p=0}^{d-1}|p\rangle|p\rangle$.

In [6]:
# Initialize the (S, N) pairs in the operator circuit
for i in range(number_S_qudits):
    qc.apply_gate(gates.get_H_gate(), [2*i+1])
    qc.apply_gate(gates.CSUM_gate(), [2*i+1, 2*i+2])

# Initialize the (S, N) pairs in the gate circuit
for i in range(number_S_qudits):
    qc_gates.apply_gate(gates.get_H_gate(), [2*i+1])
    qc_gates.apply_gate(gates.CSUM_gate(), [2*i+1, 2*i+2])

Display the quantum state

In [7]:
print("State of the operator circuit after initialization:")
qc.display_state_abs()

print("\n")

print("State of the gate circuit after initialization:")
qc_gates.display_state_abs()

print("\n")
print(f"The two states are equal: {compare_quantum_states(qc, qc_gates)}")


State of the operator circuit after initialization:
|0000000> : 0.1925+0.0000j => |0000000|=0.1925 => Prob=0.0370
|0000011> : 0.1925+0.0000j => |0000011|=0.1925 => Prob=0.0370
|0000022> : 0.1925+0.0000j => |0000022|=0.1925 => Prob=0.0370
|0001100> : 0.1925+0.0000j => |0001100|=0.1925 => Prob=0.0370
|0001111> : 0.1925+0.0000j => |0001111|=0.1925 => Prob=0.0370
|0001122> : 0.1925+0.0000j => |0001122|=0.1925 => Prob=0.0370
|0002200> : 0.1925+0.0000j => |0002200|=0.1925 => Prob=0.0370
|0002211> : 0.1925+0.0000j => |0002211|=0.1925 => Prob=0.0370
|0002222> : 0.1925+0.0000j => |0002222|=0.1925 => Prob=0.0370
|0110000> : 0.1925+0.0000j => |0110000|=0.1925 => Prob=0.0370
|0110011> : 0.1925+0.0000j => |0110011|=0.1925 => Prob=0.0370
|0110022> : 0.1925+0.0000j => |0110022|=0.1925 => Prob=0.0370
|0111100> : 0.1925+0.0000j => |0111100|=0.1925 => Prob=0.0370
|0111111> : 0.1925+0.0000j => |0111111|=0.1925 => Prob=0.0370
|0111122> : 0.1925+0.0000j => |0111122|=0.1925 => Prob=0.0370
|0112200> : 0.1925

## Construct the encryption operator

### Construct the encryption operator as given by equation (20)

In [8]:
targets = [0]+[2*i+1 for i in range(number_S_qudits)]
enc_matrix = gates.create_Uencryption(number_S_qudits)
qc.apply_gate(enc_matrix, targets=targets)

### Construct the encryption operator as described in section IV

We apply the $P_Z$ operator

In [9]:
# Apply the C(X_d) ladder
qc_gates.apply_gate(gates.CSUM_gate(), targets=[0, 1])
for i in range(number_S_qudits-1):
    qc_gates.apply_gate(gates.CSUM_gate(), targets=[2*i+1, 2*(i+1)+1])

# Apply the Q gate to the last S qudit
qc_gates.apply_gate(gates.Q_gate(), targets=[2*(number_S_qudits-1)+1])

# Apply the inverse of the C(X_d) ladder
for i in range(number_S_qudits-1, 0, -1):
    qc_gates.apply_gate(gates.CSUM_gate().conj().T, targets=[2*(i-1)+1, 2*(i)+1])
qc_gates.apply_gate(gates.CSUM_gate().conj().T, targets=[0, 1])

We apply the $P_X$ operator

In [10]:
# Apply the F gate to the qudits A and all the S qudits
qc_gates.apply_gate(gates.get_H_gate(), [0])
for i in range(number_S_qudits):
    qc_gates.apply_gate(gates.get_H_gate(), [2*i+1])

# Apply the C(X_d) ladder
qc_gates.apply_gate(gates.CSUM_gate(), targets=[0, 1])
for i in range(number_S_qudits-1):
    qc_gates.apply_gate(gates.CSUM_gate(), targets=[2*i+1, 2*(i+1)+1])

# Apply the Q gate to the last S qudit
qc_gates.apply_gate(gates.Q_gate(), targets=[2*(number_S_qudits-1)+1])

# Apply the inverse of the C(X_d) ladder
for i in range(number_S_qudits-1, 0, -1):
    qc_gates.apply_gate(gates.CSUM_gate().conj().T, targets=[2*(i-1)+1, 2*(i)+1])
qc_gates.apply_gate(gates.CSUM_gate().conj().T, targets=[0, 1])

# Apply the F^{dagger} gate to the qudits A and all the S qudits
for i in range(number_S_qudits):
    qc_gates.apply_gate(gates.get_H_gate().conj().T, [2*i+1])
qc_gates.apply_gate(gates.get_H_gate().conj().T, [0])

In [11]:
print("State of the operator circuit after encryption:")
qc.display_state_abs()

print("\n")

print("State of the gate circuit after encryption:")
qc_gates.display_state_abs()

print("\n")
print(f"The two states are equal: {compare_quantum_states(qc, qc_gates)}")

State of the operator circuit after encryption:
|0000000> : 0.0962-0.0556j => |0000000|=0.1111 => Prob=0.0123
|0000011> : 0.0962-0.0556j => |0000011|=0.1111 => Prob=0.0123
|0000022> : 0.0000+0.1111j => |0000022|=0.1111 => Prob=0.0123
|0001100> : 0.0962-0.0556j => |0001100|=0.1111 => Prob=0.0123
|0001111> : 0.0000+0.1111j => |0001111|=0.1111 => Prob=0.0123
|0001122> : 0.0962-0.0556j => |0001122|=0.1111 => Prob=0.0123
|0002200> : 0.0000+0.1111j => |0002200|=0.1111 => Prob=0.0123
|0002211> : 0.0962-0.0556j => |0002211|=0.1111 => Prob=0.0123
|0002222> : 0.0962-0.0556j => |0002222|=0.1111 => Prob=0.0123
|0110000> : 0.0962-0.0556j => |0110000|=0.1111 => Prob=0.0123
|0110011> : 0.0000+0.1111j => |0110011|=0.1111 => Prob=0.0123
|0110022> : 0.0962-0.0556j => |0110022|=0.1111 => Prob=0.0123
|0111100> : 0.0000+0.1111j => |0111100|=0.1111 => Prob=0.0123
|0111111> : 0.0962-0.0556j => |0111111|=0.1111 => Prob=0.0123
|0111122> : 0.0962-0.0556j => |0111122|=0.1111 => Prob=0.0123
|0112200> : 0.0962-0.0

## Construct the decryption operator

In [12]:
choose_pair = 3 # The index of the (S,N) pair that we choose for decryption
SN_target = [2*(choose_pair-1)+1, 2*(choose_pair-1)+2] 
N_targets = []
for i in range(choose_pair-1):
    N_targets.append(2*i+2)
for i in range(choose_pair, number_S_qudits):
    N_targets.append(2*i+2)
N_target = [N_targets[0]]

targets_dec = SN_target + N_targets
print(f"Targets for decryption: {targets_dec}")

Targets for decryption: [5, 6, 2, 4]


### Construct the encryption operator as given by equation (22)

In [13]:
dec_matrix = gates.create_Udecryption(number_S_qudits)

### Construct the decryption operator as described in section IV

In [14]:
# Apply the aditional SWAP gate needed in the circuit
qc_gates.apply_gate(gates.SWAP_gate(), SN_target)

# Apply the T_{S_1, N_1}^{dagger} gate
qc_gates.apply_gate(gates.CSUM_gate().conj().T, SN_target)
qc_gates.apply_gate(gates.get_H_gate().conj().T, [SN_target[0]])
qc_gates.apply_gate(gates.SWAP_gate(), SN_target)

# Apply each of the T_{kl} gate, with k*dim+l in {1, .., dimension^2-1}
for k in range(dimension):
    for l in range(dimension):
        if k == 0 and l == 0:
            continue
    
        # Compute the coefficients
        ck = np.exp(-1j*np.pi*(k)*(k+dimension%2)/dimension)
        cl = np.exp(-1j*np.pi*(l)*(l+dimension%2)/dimension)
        c0 = np.exp(-1j*np.pi*(0)*(0+dimension%2)/dimension)
        c_00 = (c0 * c0)**(-1)
        c_kl = (ck * cl)**(-1)

        control_qudits = SN_target
        control_levels = [k, l]

        gateI = c_kl/c_00 * np.eye(dimension, dtype=np.complex128)
        gateX = np.linalg.matrix_power(gates.get_X_gate(), k)
        gateZ = np.linalg.matrix_power(gates.get_Z_gate(), -l)
        
        # Apply the double-controlled c_kl/c_00 * I_d
        qc_gates.apply_controlled_gate(gateI, control=control_qudits, target=N_target, control_level=control_levels)

        # Apply the double-controlled X^k gate and the double-controlled Z^{-l} gate
        for i in range(1, number_S_qudits):
            target_qudits = [N_targets[i-1]]
            qc_gates.apply_controlled_gate(gateZ, control=control_qudits, target=target_qudits, control_level=control_levels)
            qc_gates.apply_controlled_gate(gateX, control=control_qudits, target=target_qudits, control_level=control_levels)

# Apply the T_{S_1, N_1} gate
qc_gates.apply_gate(gates.SWAP_gate(), SN_target)
qc_gates.apply_gate(gates.get_H_gate(), [SN_target[0]])
qc_gates.apply_gate(gates.CSUM_gate(), SN_target)


In [15]:
print("State of the operator circuit after decryption:")
qc.apply_gate(dec_matrix, targets=targets_dec)
qc.display_state_abs()

print("\n")

print("State of the gate circuit after decryption:")
# qc_gates.apply_gate(dec_matrix, targets=targets_dec)
qc_gates.display_state_abs()

print("\n")
print(f"The two states are equal: {compare_quantum_states(qc, qc_gates)}")

State of the operator circuit after decryption:
|0000000> : 0.1925+0.0000j => |0000000|=0.1925 => Prob=0.0370
|0001100> : 0.1925+0.0000j => |0001100|=0.1925 => Prob=0.0370
|0002200> : 0.1925+0.0000j => |0002200|=0.1925 => Prob=0.0370
|0110000> : 0.1925+0.0000j => |0110000|=0.1925 => Prob=0.0370
|0111100> : 0.1925+0.0000j => |0111100|=0.1925 => Prob=0.0370
|0112200> : 0.1925+0.0000j => |0112200|=0.1925 => Prob=0.0370
|0220000> : 0.1925+0.0000j => |0220000|=0.1925 => Prob=0.0370
|0221100> : 0.1925+0.0000j => |0221100|=0.1925 => Prob=0.0370
|0222200> : 0.1925+0.0000j => |0222200|=0.1925 => Prob=0.0370
|1000001> : 0.1925+0.0000j => |1000001|=0.1925 => Prob=0.0370
|1001101> : 0.1925+0.0000j => |1001101|=0.1925 => Prob=0.0370
|1002201> : 0.1925+0.0000j => |1002201|=0.1925 => Prob=0.0370
|1110001> : 0.1925+0.0000j => |1110001|=0.1925 => Prob=0.0370
|1111101> : 0.1925+0.0000j => |1111101|=0.1925 => Prob=0.0370
|1112201> : 0.1925+0.0000j => |1112201|=0.1925 => Prob=0.0370
|1220001> : 0.1925+0.0